# Regression Application [Bengaluru House pricing]

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

In [ ]:
data = pd.read_csv("Bengaluru_House_Data.csv")

In [ ]:
data.head(10)

In [ ]:
data.shape

In [ ]:
data.describe()

In [ ]:
for c in data.columns:
    print(data[c].value_counts())
    print("--------------------------")

## Data processing

In [ ]:
data.isna().sum()

In [ ]:
data.info()

In [ ]:
# Not critical data
data.drop(columns=["area_type", "availability", "society", "balcony"], inplace=True)

In [ ]:
data

In [ ]:
data.describe()

In [ ]:
data["total_sqft"].unique()

In [ ]:
def uniformRange(x):
    temp = x.split("-")
    if len(temp) == 2:
        return (float(temp[0]) + float(temp[1])) / 2
    try:
        return float(x)
    except:
        return None

In [ ]:
data["total_sqft"] = data["total_sqft"].apply(uniformRange)

In [ ]:
data["total_sqft"].unique()

In [ ]:
data.head()

In [ ]:
data["size"].value_counts()

In [ ]:
data["size"] = data["size"].fillna("2 BHK")

In [ ]:
data["location"].value_counts()

In [ ]:
data["location"] = data["location"].fillna("Sarjapur  Road")

In [ ]:
data["total_sqft"].value_counts()

In [ ]:
data["bath"] = data["bath"].fillna(data["bath"].median())

In [ ]:
# Bedroom-Hall-Kitchen (BHK)
data["BHK"] = data["size"].str.split().str.get(0).astype(int)

In [ ]:
data.head(10)

In [ ]:
data[data["BHK"] > 20]

In [ ]:
data["Price_Per_sqft"] = data["price"] * 100000 / data["total_sqft"]

In [ ]:
data["Price_Per_sqft"]

In [ ]:
data.describe()

In [ ]:
data.isna().sum()

In [ ]:
data["location"].value_counts()

In [ ]:
# Removing whitespace from location values
data["location"] = data["location"].apply(lambda x: x.strip())

In [ ]:
loc_count = data["location"].value_counts()
loc_count

In [ ]:
loc_count_less_10 = loc_count[loc_count <= 10]
len(loc_count_less_10)

In [ ]:
data["location"] = data["location"].apply(
    lambda x: "Other" if x in loc_count_less_10 else x
)

In [ ]:
data["location"].value_counts()

## Outlier Detection and Removal


In [ ]:
data.describe()

In [ ]:
(data["total_sqft"] / data["BHK"]).describe()

In [ ]:
# remove the values under the accepted area for each room
data = data[(data["total_sqft"] / data["BHK"]) >= 300]

In [ ]:
data.describe()

In [ ]:
data.shape

In [ ]:
data.isna().sum()

In [ ]:
data["Price_Per_sqft"].describe()

In [ ]:
data.reset_index(inplace=True)

In [ ]:
def remove_outliers_sqft(df):
    def filter_outliers(subdf):
        m = subdf["Price_Per_sqft"].mean()
        st = subdf["Price_Per_sqft"].std()
        return subdf[
            (subdf["Price_Per_sqft"] >= m - st) & (subdf["Price_Per_sqft"] <= m + st)
        ]
    return df.groupby("location").apply(filter_outliers).reset_index(drop=True)


data = remove_outliers_sqft(data)
data.describe()

In [ ]:
def bhk_outlier_remover(df):
    group_cols = ['location', 'BHK']
    means = df.groupby(group_cols)['Price_Per_sqft'].transform('mean')
    stds = df.groupby(group_cols)['Price_Per_sqft'].transform('std')
    counts = df.groupby(group_cols)['Price_Per_sqft'].transform('count')
    
    mask_outliers = (counts > 5) & (df['Price_Per_sqft'] < (means - stds))
    return df[~mask_outliers]



In [ ]:
data = bhk_outlier_remover(data)
data

In [ ]:
data.describe()

In [ ]:
data.shape

In [ ]:
data

In [ ]:
data.drop(columns=["size",'Price_Per_sqft', 'index'], inplace=True)

### Cleaned Data

In [ ]:
data.head()

In [ ]:
data.to_csv('Cleaned_data.csv')

In [ ]:
X = data.drop(columns = 'price')
y = data['price']

### Splitting data

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

#Splitted Data
print('X_train shape is ' , X_train.shape)
print('X_test shape is ' , X_test.shape)
#print('y_train shape is ' , y_train.shape)
#print('y_test shape is ' , y_test.shape)

## Applying Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn. compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score

In [ ]:
column_trans = make_column_transformer((OneHotEncoder(sparse_output=False), ['location']),remainder='passthrough')

In [ ]:
scaler = StandardScaler()

lr =  LinearRegression(fit_intercept=True,copy_X=True,n_jobs=-1)

pipe = make_pipeline(column_trans, scaler, lr)


In [ ]:

pipe.fit(X_train,y_train)


In [ ]:

y_pred_lr = pipe.predict(X_test)
LReg = r2_score(y_test, y_pred_lr)

In [ ]:
lasso = Lasso()

In [ ]:
pipe = make_pipeline(column_trans, scaler, lasso)

In [ ]:
pipe.fit(X_train, y_train)

In [ ]:
y_predect_lasso = pipe.predict(X_test)
l = r2_score(y_test,y_predect_lasso)

In [ ]:
ridge = Ridge()

In [ ]:
pipe = make_pipeline(column_trans, scaler, ridge)

In [ ]:
pipe.fit(X_train, y_train)

In [ ]:
y_predect_ridge = pipe.predict(X_test)
r = r2_score(y_test,y_predect_ridge)

In [ ]:
print("Linear Regression ", LReg)
print("Lasso ", l)
print("Ridge ", r)

In [ ]:
import pickle

In [ ]:
pickle.dump(pipe, open('RigeModel.pkl','wb'))